# Manutenção Preditiva Industrial
### Análise de dados e classificação de falhas mecânicas

**Autor:** [Bruno](https://github.com/CorreaBrunoMiguel)  
**Atividade:** Desenvolvimento de IA para Análise Preditiva [T3]  
**Etapa acadêmica:** Módulo 1 — Projeto Avaliativo  
**Tecnologias:** Python e Jupyter Notebook  

---

## Introdução

A identificação de falhas em equipamentos industriais pode apoiar o planejamento da manutenção e a redução de paradas não programadas. Nesse contexto, este projeto utiliza uma base histórica com medidas de temperatura, velocidade de rotação, torque, desgaste da ferramenta e categoria do equipamento para investigar a classificação de seu estado operacional.

O objetivo é desenvolver e comparar dois classificadores — K-Nearest Neighbors (KNN) e Árvore de Decisão — considerando a variável alvo `falha_maquina`, em que **0 representa funcionamento normal** e **1 representa ocorrência de falha**.

O trabalho será organizado em sete fases: análise exploratória, limpeza e tratamento dos dados, engenharia de atributos, divisão e balanceamento, escalonamento, ajuste de hiperparâmetros e avaliação final. As escolhas técnicas serão documentadas junto às evidências obtidas, permitindo acompanhar o raciocínio da análise e verificar os resultados.

A comparação exigida pelo projeto utilizará a acurácia nos conjuntos de treino e teste, com discussão dos indícios de sobreajuste e das limitações do experimento.

## Fontes do projeto

- **Base de dados:** [manutencao_preditiva.csv](../data/manutencao_preditiva.csv).
- **Enunciado oficial:** [requisitos e critérios de avaliação](../docs/enunciado-projeto.md).
- **Anotações do Departamento de Engenharia:** [descrição das colunas e restrições de uso na modelagem](../docs/referencias/anotacoes-engenharia.docx).

## Fase 1 — Análise exploratória dos dados (EDA)

### Inspeção inicial da base

A inspeção inicial busca conhecer a estrutura dos dados e identificar características relevantes para a modelagem. Para isso, são examinadas uma amostra dos registros, as dimensões da base, os tipos das colunas e as estatísticas descritivas. Também são investigados o padrão inicial de ausências e a distribuição dos tipos de produto e da variável alvo.

Os resultados desta seção descrevem a base original, anterior aos procedimentos de limpeza e transformação.

In [1]:
import pandas as pd

from pathlib import Path

print("Pandas:", pd.__version__)
print("\n ========== \n")
print("Diretório atual:", Path.cwd())

Pandas: 3.0.5


Diretório atual: /home/bruno/Desktop/senai/projeto_final/notebooks


In [2]:
df_origin = pd.read_csv("../data/manutencao_preditiva.csv", index_col="udi")
df = df_origin.copy()

df.head()

,id_produto,tipo,temperatura_ar_k,temperatura_processo_k,velocidade_rotacao_rpm,torque_nm,desgaste_ferramenta_min,falha_maquina,falha_twf,falha_hdf,falha_pwf,falha_osf,falha_rnf
udi,,,,,,,,,,,,,
1,M14860,M,298.1,308.6,1551.0,42.8,0,0,0,0,0,0,0
2,L47181,L,298.2,308.7,1408.0,46.3,3,0,0,0,0,0,0
3,L47182,L,298.1,308.5,1498.0,49.4,5,0,0,0,0,0,0
4,L47183,L,NaN,NaN,NaN,NaN,7,0,0,0,0,0,0
5,L47184,L,298.2,308.7,1408.0,40.0,9,0,0,0,0,0,0


### Papel das colunas

Conforme as anotações do Departamento de Engenharia, as colunas possuem funções distintas na análise:

| Grupo | Colunas | Função |
| --- | --- | --- |
| Identificadores | `udi`, `id_produto` | Identificação dos registros e produtos. |
| Categoria do produto | `tipo` | Classificação do produto nas categorias L, M e H. |
| Medidas operacionais | Temperaturas, velocidade de rotação, torque e desgaste da ferramenta | Caracterização das condições operacionais; candidatas a variáveis preditoras. |
| Variável alvo | `falha_maquina` | Funcionamento normal (0) ou ocorrência de falha (1). |
| Indicadores históricos de falha | `falha_twf`, `falha_hdf`, `falha_pwf`, `falha_osf`, `falha_rnf` | Informações históricas com uso restrito pela engenharia. |

As temperaturas estão expressas em kelvin, a rotação em rpm, o torque em N·m e o desgaste da ferramenta em minutos. Os identificadores representam códigos, mesmo quando armazenados como números.

Por determinação da engenharia, os cinco indicadores históricos de falha serão excluídos das entradas dos modelos e não serão utilizados na construção de atributos derivados.

In [3]:
linhas, colunas = df.shape

print(f"""
Dataset possui:
    - Linhas: {linhas}
    - Colunas: {colunas}
""")

print("========== \n")

display(df.dtypes)


Dataset possui:
    - Linhas: 10000
    - Colunas: 13




id_produto                     str
tipo                           str
temperatura_ar_k           float64
temperatura_processo_k     float64
velocidade_rotacao_rpm     float64
torque_nm                  float64
desgaste_ferramenta_min      int64
falha_maquina                int64
falha_twf                    int64
falha_hdf                    int64
falha_pwf                    int64
falha_osf                    int64
falha_rnf                    int64
dtype: object

In [4]:
display(df.describe())

,temperatura_ar_k,temperatura_processo_k,velocidade_rotacao_rpm,torque_nm,desgaste_ferramenta_min,falha_maquina,falha_twf,falha_hdf,falha_pwf,falha_osf,falha_rnf
count,9500.000000,9500.000000,9500.000000,9500.000000,10000.000000,10000.000000,10000.000000,10000.000000,10000.000000,10000.000000,10000.00000
mean,300.002158,310.000895,1539.245263,39.974168,107.951000,0.033900,0.004600,0.011500,0.009500,0.009800,0.00190
std,2.001689,1.486432,180.273589,9.995453,63.654147,0.180981,0.067671,0.106625,0.097009,0.098514,0.04355
min,295.300000,305.700000,1168.000000,3.800000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.00000
25%,298.300000,308.800000,1423.000000,33.100000,53.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.00000
50%,300.100000,310.100000,1504.000000,40.100000,108.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.00000
75%,301.500000,311.100000,1613.000000,46.700000,162.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.00000
max,304.500000,313.800000,2886.000000,76.600000,253.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.00000


In [5]:
col_dados_faltantes = [
    "temperatura_ar_k",
    "temperatura_processo_k",
    "velocidade_rotacao_rpm",
    "torque_nm",
]

display(df[col_dados_faltantes].isna().sum(axis=1).value_counts())

0    9500
4     500
Name: count, dtype: int64

### Padrão inicial de valores ausentes

O resumo estatístico revelou 500 valores ausentes em cada uma das quatro medidas: temperatura do ar, temperatura do processo, velocidade de rotação e torque.

A verificação por linha confirmou que essas ausências coincidem nos mesmos 500 registros, equivalentes a 5% da base. Nos outros 9.500 registros, as quatro medidas estão presentes. Assim, o padrão observado é de ausência conjunta dessas medidas.

A variável alvo está preenchida nos 10.000 registros. O tratamento das ausências e sua justificativa serão apresentados na Fase 2 — Limpeza e tratamento dos dados.

In [6]:
total = len(df)
sem_falhas = (df["falha_maquina"] == 0).sum()
com_falhas = (df["falha_maquina"] == 1).sum()

resumo = pd.DataFrame(
    {
        "Total de registros": [total],
        "Sem falhas": [sem_falhas],
        "% Sem falhas": [sem_falhas / total],
        "Com falhas": [com_falhas],
        "% Com falhas": [com_falhas / total],
    }
)

display(
    resumo.style.hide(axis="index").format(
        {
            "Total de registros": "{:,}",
            "Sem falhas": "{:,}",
            "% Sem falhas": "{:.2%}",
            "Com falhas": "{:,}",
            "% Com falhas": "{:.2%}",
        }
    )
)

print("\n========== \n")

tipos = pd.DataFrame(
    {
        "Quantidade": df["tipo"].value_counts(dropna=False),
        "Percentual_Base": df["tipo"].value_counts(normalize=True, dropna=False),
    }
).reset_index()
tipos["qtd_falhas"] = tipos["tipo"].map(df.groupby("tipo")["falha_maquina"].sum())
tipos["falhas_%"] = tipos.qtd_falhas / tipos.Quantidade
display(
    tipos.style.hide(axis="index").format(
        {"Percentual_Base": "{:.2%}", "falhas_%": "{:.2%}"}
    )
)

Total de registros,Sem falhas,% Sem falhas,Com falhas,% Com falhas
"10,000","9,661",96.61%,339,3.39%


tipo,Quantidade,Percentual_Base,qtd_falhas,falhas_%
L,6000,60.00%,235,3.92%
M,2997,29.97%,83,2.77%
H,1003,10.03%,21,2.09%


### Distribuição da variável alvo

A variável `falha_maquina` representa funcionamento normal (0) ou ocorrência de falha (1). Dos 10.000 registros, 9.661 correspondem à classe normal (96,61%) e 339 à classe de falha (3,39%).

Essa distribuição caracteriza um desbalanceamento de classes, com as falhas constituindo a classe minoritária. A predominância de registros normais exige cuidado na interpretação da acurácia: um valor elevado, isoladamente, pode ocultar dificuldade em identificar falhas.

## Fase 2 — Limpeza e tratamento dos dados

## Fase 3 — Engenharia de atributos

## Fase 4 — Divisão e balanceamento dos dados

## Fase 5 — Escalonamento de variáveis

## Fase 6 — Ajuste de hiperparâmetros e análise de overfitting

## Fase 7 — Avaliação da acurácia e veredito final